In [ ]:
import numpy as np

np.random.seed(2026)

# Datos del problema
N = 980       # total de estampas del album
S = 7         # estampas por sobre
R = 200       # numero de simulaciones
PS = 9.5      # precio del sobre
PC = 975      # precio de la caja
SC = 104      # sobres por caja
PRESUPUESTO = 2000

def abrir_sobre(album):
    estampas = np.random.randint(0, N, S)
    for estampa in estampas:
        album[estampa] += 1

def simular_album(estrategia):
    album = np.zeros(N, dtype=int)
    costo = 0
    costo_al_llegar_a_50 = None

    while np.sum(album > 0) < N:
        faltan = N - np.sum(album > 0)

        if faltan <= 50 and costo_al_llegar_a_50 is None:
            costo_al_llegar_a_50 = costo

        if estrategia == 'sobres':
            sobres = 1
            costo += PS
        elif estrategia == 'cajas':
            sobres = SC
            costo += PC
        else:
            if faltan > 50:
                sobres = SC
                costo += PC
            else:
                sobres = 1
                costo += PS

        for i in range(sobres):
            abrir_sobre(album)

    repetidas = np.sum(album) - N
    costo_ultimas_50 = costo - costo_al_llegar_a_50
    return costo, repetidas, costo_ultimas_50

def promedio(estrategia):
    resultados = []
    for i in range(R):
        resultados.append(simular_album(estrategia))
    return np.mean(resultados, axis=0)

In [ ]:
# Problema 1
# ¿Cuál es la estrategia más barata para completar el álbum: comprar solo sobres, solo cajas o una combinación de ambos?

estrategias = ['sobres', 'cajas', 'combinacion']
costos = {}

for estrategia in estrategias:
    costo_promedio, repetidas, extra = promedio(estrategia)
    costos[estrategia] = costo_promedio
    print(estrategia, ': Q', round(costo_promedio, 2))

mejor = min(costos, key=costos.get)
print('\nRespuesta: la estrategia mas barata es', mejor)


In [ ]:
# Problema 2
# ¿Cuál es la probabilidad de completar al menos el 90% del álbum con un presupuesto fijo de Q2,000?

def simular_presupuesto():
    album = np.zeros(N, dtype=int)
    dinero = PRESUPUESTO

    while dinero >= PS:
        if dinero >= PC:
            dinero -= PC
            sobres = SC
        else:
            dinero -= PS
            sobres = 1

        for _ in range(sobres):
            abrir_sobre(album)

    porcentaje = np.sum(album > 0) / N
    return porcentaje >= 0.90

exitos = [simular_presupuesto() for i in range(R)]
probabilidad = np.mean(exitos)

print('Respuesta: la probabilidad es', round(probabilidad * 100, 2), '%')


In [ ]:
# Problema 3
# ¿Cuántas estampas repetidas se acumulan en promedio al completar el álbum?

costo_promedio, repetidas_promedio, extra = promedio('combinacion')

print('Respuesta: se acumulan en promedio', round(repetidas_promedio), 'estampas repetidas')


In [ ]:
# Problema 4
# ¿Cuánto dinero adicional cuesta conseguir las últimas 50 estampas del álbum?

costo_promedio, repetidas, extra_promedio = promedio('combinacion')

print('Respuesta: las ultimas 50 estampas cuestan aproximadamente Q', round(extra_promedio, 2))


In [ ]:
# Problema 5
# ¿Qué pasa si dos personas llenan el álbum juntas e intercambian todas sus repetidas?

def simular_dos_personas():
    album = np.zeros(N, dtype=int)
    costo = 0

    while np.min(album) < 2:
        abrir_sobre(album)
        costo += PS

    repetidas = np.sum(album) - (2 * N)
    return costo, repetidas

resultados = [simular_dos_personas() for i in range(R)]
costo_total, repetidas = np.mean(resultados, axis=0)

print('Respuesta: entre dos personas gastan aproximadamente Q', round(costo_total, 2))
print('Cada persona paga aproximadamente Q', round(costo_total / 2, 2))
print('Juntas acumulan aproximadamente', round(repetidas), 'repetidas')
